# Pharma RAG Demo — Support Instructor (Module 8A)

> **MOCK — for educational demonstration only, not medical advice.**

Same RAG machinery as the M8A integration lab, applied to a small (50-doc) mock pharmaceutical product information corpus. The domain shift makes the **groundedness** metric obvious: a clinical claim like *"contraindicated in pregnancy"* must come from the retrieved context, not be invented by the generator.

**Demo flow (~30 min):**
1. Frame + safety disclaimer
2. Build + ingest the 50-doc mock corpus
3. Single-fact extractive queries (RAG works cleanly)
4. Synthesis query (multi-field grounded answer)
5. Borderline / out-of-corpus queries (model should abstain)
6. Pivot to the integration lab

## Start the Weaviate Docker container

Spins up the demo-specific Weaviate instance on port **8091** (non-conflicting with anything the learner runs locally). Idempotent: if a container named `weaviate-si-demo-int` already exists, it's started rather than re-run; if it's already running, this is a no-op. The final loop blocks until `/v1/.well-known/ready` returns 200 so the rest of the notebook can assume Weaviate is up.

In [ ]:
import subprocess      # runs shell/Docker commands from Python
import time            # provides sleep() for the Weaviate readiness polling loop
import urllib.request  # makes HTTP requests to check Weaviate's health endpoint

CONTAINER_NAME = "weaviate-si-demo-int"           # unique Docker container name for this demo
WEAVIATE_IMAGE = "semitechnologies/weaviate:1.28.4"  # pinned image version for reproducibility
HOST_PORT = 8091   # host port mapped to Weaviate's internal REST port 8080; avoids conflicts with other local services
GRPC_PORT = 50051  # gRPC port required by the Weaviate v4 Python client for fast data transfer

def _run(cmd: list[str]) -> str:
    # runs a shell command and returns its stdout as a trimmed string; errors are ignored (check=False)
    return subprocess.run(cmd, capture_output=True, text=True, check=False).stdout.strip()

# query Docker for all containers (running or stopped) matching the container name
existing = _run(["docker", "ps", "-a", "--filter", f"name=^{CONTAINER_NAME}$", "--format", "{{.Names}}"])
# query Docker for only currently running containers matching the container name
running  = _run(["docker", "ps",      "--filter", f"name=^{CONTAINER_NAME}$", "--format", "{{.Names}}"])

if running == CONTAINER_NAME:
    # container is already running — nothing to do, avoid a duplicate-run error
    print(f"Container {CONTAINER_NAME} is already running.")
elif existing == CONTAINER_NAME:
    # container exists but is stopped — restart it to reuse its existing data volume
    print(f"Starting existing container {CONTAINER_NAME}…")
    subprocess.run(["docker", "start", CONTAINER_NAME], check=True)
else:
    # container doesn't exist yet — create and start a fresh one
    print(f"Creating and starting container {CONTAINER_NAME}…")
    subprocess.run([
        "docker", "run", "-d",                          # -d: run in detached (background) mode
        "--name", CONTAINER_NAME,                        # assign a stable name so later cells can reference it
        "-p", f"{HOST_PORT}:8080",                       # map host 8091 → container's REST API port 8080
        "-p", f"{GRPC_PORT}:50051",                      # expose the gRPC port so the v4 client can connect
        "-e", "DEFAULT_VECTORIZER_MODULE=none",          # disable built-in vectorizers; we supply our own vectors
        "-e", "ENABLE_MODULES=",                         # no extra Weaviate modules needed for this demo
        "-e", "PERSISTENCE_DATA_PATH=/var/lib/weaviate", # directory inside the container where data is stored
        WEAVIATE_IMAGE,                                  # the pinned Weaviate Docker image to run
    ], check=True)

ready_url = f"http://localhost:{HOST_PORT}/v1/.well-known/ready"  # Weaviate's standard readiness endpoint
for attempt in range(30):  # poll up to 30 times (~30 seconds) before giving up
    try:
        with urllib.request.urlopen(ready_url, timeout=1) as resp:
            if resp.status == 200:  # HTTP 200 means Weaviate is fully initialised and ready for requests
                print(f"Weaviate ready at {ready_url}")
                break
    except Exception:
        pass  # connection refused or timeout — Weaviate is still initialising, try again
    time.sleep(1)  # wait one second between poll attempts
else:
    # the for-loop completed all 30 attempts without a break — Weaviate never became ready
    raise RuntimeError(f"Weaviate did not become ready at {ready_url} within 30s.")

## 0. Imports + module-level model loading

We load `flan-t5-base` and `all-MiniLM-L6-v2` **once at module level** so subsequent calls don't pay the load cost. CPU inference is ~1–3 s/call when loaded correctly; ~30 s+ if reloaded per call.

In [ ]:
import json       # serialises/deserialises records to/from the JSONL corpus file
import os         # available for environment-variable and path operations if needed
from pathlib import Path  # provides cross-platform, object-oriented file path handling

import weaviate   # Weaviate v4 Python client — connects to the vector database
from weaviate.classes.config import Configure, Property, DataType  # schema-definition helpers for collection creation
from weaviate.classes.query import MetadataQuery  # requests extra per-result metadata (e.g. cosine distance) in queries

from sentence_transformers import SentenceTransformer           # lightweight model for computing sentence-level embeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM   # Hugging Face classes for loading a seq2seq language model

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # small 384-dim model; fast on CPU, good retrieval quality
GEN_MODEL_NAME   = "google/flan-t5-base"                       # instruction-tuned T5; used as the RAG answer generator
WEAVIATE_HOST    = "localhost"                                  # Weaviate is running locally inside Docker
WEAVIATE_PORT    = 8091                                        # host port set in the start-container cell above
CLASS_NAME       = "Post"                                       # Weaviate collection name; matches the M8A lab schema
CORPUS_PATH      = Path("pharma_corpus.jsonl")                 # output file path for the mock pharmaceutical corpus

print("Loading embedder…")
embedder      = SentenceTransformer(EMBED_MODEL_NAME)           # load embedder into memory once; reused for all encode() calls
print("Loading generator…")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)           # tokenizer converts text to token IDs for the model
gen_model     = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)   # load generator weights; stays in memory for all queries
print("Models ready.")  # signals that both models are fully loaded before any pipeline calls are made

## 1. Author the 50-drug mock corpus

Each entry has the four DailyMed-style fields (indication, dosage, contraindications, warnings) plus a mandatory `notice` field marking it as MOCK content. **These summaries are instructor-curated for the demo and are not vetted medical information.**

In [ ]:
MOCK_NOTICE = "MOCK — for educational demonstration only, not medical advice"  # disclaimer appended to every document

# Each dict represents one drug and has four DailyMed-style fields:
#   indication       — what the drug is used for
#   dosage           — typical adult dose and schedule
#   contraindications — patient groups or co-medications that must avoid it
#   warnings         — safety risks, boxed warnings, and monitoring requirements
# A "notice" field is added at the end of this cell to every entry.
docs = [
    {"id": "pharma:001", "title": "Acetaminophen — adult tablets",
     "indication": "Mild to moderate pain; fever reduction.",
     "dosage": "Adults: 500–1000 mg every 4–6 hours, maximum 4 g/day.",
     "contraindications": "Severe hepatic impairment; alcohol use disorder.",
     "warnings": "Risk of hepatotoxicity at doses above 4 g/day. Many combination cold/flu products contain acetaminophen — check labels."},
    {"id": "pharma:002", "title": "Ibuprofen — adult tablets",
     "indication": "Pain, fever, and inflammation.",
     "dosage": "Adults: 200–400 mg every 4–6 hours, maximum 1.2 g/day OTC.",
     "contraindications": "Active GI bleeding; severe heart failure; third-trimester pregnancy.",
     "warnings": "Increased risk of cardiovascular events and GI bleeding with prolonged use. Avoid concomitant NSAIDs."},
    {"id": "pharma:003", "title": "Aspirin — low-dose",
     "indication": "Secondary prevention of cardiovascular events.",
     "dosage": "Adults: 81 mg once daily.",
     "contraindications": "Active peptic ulcer; bleeding disorders; children with viral illness (Reye syndrome risk).",
     "warnings": "Bleeding risk increases with anticoagulants. Discontinue before elective surgery per clinician guidance."},
    {"id": "pharma:004", "title": "Naproxen — adult tablets",
     "indication": "Pain, dysmenorrhea, arthritis.",
     "dosage": "Adults: 250–500 mg twice daily, maximum 1 g/day.",
     "contraindications": "Active GI bleeding; severe renal impairment; third-trimester pregnancy.",
     "warnings": "Cardiovascular and GI risks similar to other NSAIDs. Monitor renal function in elderly."},
    {"id": "pharma:005", "title": "Lisinopril — oral tablets",
     "indication": "Hypertension; heart failure; post-MI.",
     "dosage": "Adults: initial 10 mg once daily; maintenance 20–40 mg/day.",
     "contraindications": "Pregnancy (all trimesters); history of angioedema; bilateral renal artery stenosis.",
     "warnings": "Avoid potassium supplements and potassium-sparing diuretics. Dry cough is common. Monitor renal function and serum potassium."},
    {"id": "pharma:006", "title": "Metformin — immediate-release",
     "indication": "Type 2 diabetes mellitus.",
     "dosage": "Adults: start 500 mg once or twice daily with meals; titrate to maximum 2 g/day.",
     "contraindications": "eGFR < 30 mL/min/1.73 m²; metabolic acidosis.",
     "warnings": "Rare risk of lactic acidosis. Hold prior to iodinated contrast in at-risk patients."},
    {"id": "pharma:007", "title": "Atorvastatin — oral tablets",
     "indication": "Hyperlipidemia; cardiovascular risk reduction.",
     "dosage": "Adults: 10–80 mg once daily.",
     "contraindications": "Active liver disease; pregnancy; breastfeeding.",
     "warnings": "Risk of myopathy, especially with concomitant fibrates or CYP3A4 inhibitors. Monitor LFTs if symptomatic."},
    {"id": "pharma:008", "title": "Amlodipine — oral tablets",
     "indication": "Hypertension; chronic stable angina.",
     "dosage": "Adults: 5–10 mg once daily.",
     "contraindications": "Severe aortic stenosis; cardiogenic shock.",
     "warnings": "Peripheral edema is common. Reflex tachycardia possible at higher doses."},
    {"id": "pharma:009", "title": "Metoprolol — succinate ER",
     "indication": "Hypertension; heart failure; angina.",
     "dosage": "Adults: 25–200 mg once daily.",
     "contraindications": "Severe bradycardia; high-degree AV block; decompensated heart failure.",
     "warnings": "Do not discontinue abruptly. May mask hypoglycemia in diabetes. Caution in asthma/COPD."},
    {"id": "pharma:010", "title": "Losartan — oral tablets",
     "indication": "Hypertension; diabetic nephropathy.",
     "dosage": "Adults: 25–100 mg once daily.",
     "contraindications": "Pregnancy; concomitant aliskiren in diabetes.",
     "warnings": "Risk of hyperkalemia. Monitor renal function and electrolytes."},
    {"id": "pharma:011", "title": "Warfarin — oral tablets",
     "indication": "Anticoagulation for atrial fibrillation, VTE, mechanical valves.",
     "dosage": "Adults: individualized to target INR (typically 2–3); start 2–5 mg/day.",
     "contraindications": "Active bleeding; pregnancy; recent neurosurgery.",
     "warnings": "Narrow therapeutic index. Many drug and food interactions (vitamin K, antibiotics, NSAIDs). Monitor INR regularly."},
    {"id": "pharma:012", "title": "Clopidogrel — oral tablets",
     "indication": "Antiplatelet therapy in ACS and post-stent.",
     "dosage": "Adults: 75 mg once daily; 300–600 mg loading dose in ACS.",
     "contraindications": "Active pathological bleeding.",
     "warnings": "Reduced effectiveness in CYP2C19 poor metabolizers. Hold 5 days before elective surgery."},
    {"id": "pharma:013", "title": "Furosemide — oral tablets",
     "indication": "Edema; hypertension.",
     "dosage": "Adults: 20–80 mg once or twice daily; titrate to response.",
     "contraindications": "Anuria; severe electrolyte depletion.",
     "warnings": "Risk of hypokalemia, hyponatremia, ototoxicity at high IV doses."},
    {"id": "pharma:014", "title": "Hydrochlorothiazide — oral tablets",
     "indication": "Hypertension; mild edema.",
     "dosage": "Adults: 12.5–25 mg once daily.",
     "contraindications": "Anuria; sulfonamide hypersensitivity.",
     "warnings": "Hypokalemia, hyperuricemia, photosensitivity. Increased non-melanoma skin cancer risk with long-term use."},
    {"id": "pharma:015", "title": "Omeprazole — delayed-release",
     "indication": "GERD; peptic ulcer disease.",
     "dosage": "Adults: 20–40 mg once daily before breakfast.",
     "contraindications": "Concomitant rilpivirine.",
     "warnings": "Long-term use linked to B12 deficiency, hypomagnesemia, fracture risk. Reduces clopidogrel activation."},
    {"id": "pharma:016", "title": "Pantoprazole — oral tablets",
     "indication": "GERD; erosive esophagitis.",
     "dosage": "Adults: 40 mg once daily.",
     "contraindications": "Hypersensitivity to benzimidazoles.",
     "warnings": "Similar long-term risks to other PPIs. Acute interstitial nephritis reported rarely."},
    {"id": "pharma:017", "title": "Ranitidine — oral tablets",
     "indication": "Heartburn; peptic ulcer (historical).",
     "dosage": "Adults: 150 mg twice daily (mock product info — most products withdrawn).",
     "contraindications": "Hypersensitivity.",
     "warnings": "Many ranitidine products were withdrawn due to NDMA impurity concerns. Listed here for demo completeness only."},
    {"id": "pharma:018", "title": "Famotidine — oral tablets",
     "indication": "Heartburn; GERD; peptic ulcer.",
     "dosage": "Adults: 20 mg twice daily or 40 mg at bedtime.",
     "contraindications": "Hypersensitivity.",
     "warnings": "Adjust dose in renal impairment. CNS effects (confusion) in elderly."},
    {"id": "pharma:019", "title": "Loratadine — oral tablets",
     "indication": "Seasonal and perennial allergic rhinitis; urticaria.",
     "dosage": "Adults: 10 mg once daily.",
     "contraindications": "Hypersensitivity.",
     "warnings": "Generally non-sedating. Reduce dose in hepatic impairment."},
    {"id": "pharma:020", "title": "Cetirizine — oral tablets",
     "indication": "Allergic rhinitis; chronic idiopathic urticaria.",
     "dosage": "Adults: 10 mg once daily.",
     "contraindications": "End-stage renal disease on hemodialysis.",
     "warnings": "May cause drowsiness. Avoid alcohol."},
    {"id": "pharma:021", "title": "Diphenhydramine — oral tablets",
     "indication": "Allergy; short-term insomnia; motion sickness.",
     "dosage": "Adults: 25–50 mg every 4–6 hours.",
     "contraindications": "Narrow-angle glaucoma; urinary retention; neonates.",
     "warnings": "Strongly anticholinergic. Avoid in elderly (Beers criteria). Marked sedation."},
    {"id": "pharma:022", "title": "Albuterol — inhalation aerosol",
     "indication": "Acute bronchospasm in asthma and COPD.",
     "dosage": "Adults: 1–2 inhalations every 4–6 hours as needed.",
     "contraindications": "Hypersensitivity to albuterol.",
     "warnings": "Excessive use signals worsening control. Paradoxical bronchospasm possible. Monitor for tachycardia."},
    {"id": "pharma:023", "title": "Fluticasone — inhalation powder",
     "indication": "Maintenance treatment of asthma.",
     "dosage": "Adults: 100–500 mcg twice daily.",
     "contraindications": "Primary treatment of status asthmaticus.",
     "warnings": "Oral candidiasis; hoarseness. Rinse mouth after use. Long-term high-dose risk of systemic corticosteroid effects."},
    {"id": "pharma:024", "title": "Montelukast — oral tablets",
     "indication": "Asthma maintenance; allergic rhinitis.",
     "dosage": "Adults: 10 mg once daily in the evening.",
     "contraindications": "Hypersensitivity.",
     "warnings": "Boxed warning for serious neuropsychiatric events (agitation, depression, suicidal thoughts)."},
    {"id": "pharma:025", "title": "Amoxicillin — oral capsules",
     "indication": "Susceptible bacterial infections (otitis, sinusitis, UTI).",
     "dosage": "Adults: 500 mg every 8 hours or 875 mg every 12 hours.",
     "contraindications": "Penicillin or beta-lactam hypersensitivity.",
     "warnings": "Risk of anaphylaxis. Rash common in mononucleosis. C. difficile-associated diarrhea possible."},
    {"id": "pharma:026", "title": "Azithromycin — oral tablets",
     "indication": "Respiratory and skin infections; STIs.",
     "dosage": "Adults: 500 mg on day 1, then 250 mg daily for 4 days (Z-pak).",
     "contraindications": "History of cholestatic jaundice with macrolides.",
     "warnings": "QT prolongation; caution with other QT-prolonging drugs. Hepatotoxicity reported."},
    {"id": "pharma:027", "title": "Ciprofloxacin — oral tablets",
     "indication": "Complicated UTI; certain GI and respiratory infections.",
     "dosage": "Adults: 250–750 mg every 12 hours.",
     "contraindications": "Concomitant tizanidine; fluoroquinolone hypersensitivity.",
     "warnings": "Boxed warnings: tendinitis/tendon rupture, peripheral neuropathy, CNS effects, exacerbation of myasthenia gravis. Avoid in pediatrics unless no alternative."},
    {"id": "pharma:028", "title": "Doxycycline — oral capsules",
     "indication": "Acne; respiratory infections; tick-borne illness.",
     "dosage": "Adults: 100 mg twice daily.",
     "contraindications": "Pregnancy; children under 8 years.",
     "warnings": "Photosensitivity; esophageal irritation — take with water and remain upright. Permanent tooth discoloration in young children."},
    {"id": "pharma:029", "title": "Cephalexin — oral capsules",
     "indication": "Skin and soft tissue infections; UTI.",
     "dosage": "Adults: 250–500 mg every 6 hours.",
     "contraindications": "Cephalosporin hypersensitivity.",
     "warnings": "Cross-reactivity possible in severe penicillin allergy. C. difficile diarrhea risk."},
    {"id": "pharma:030", "title": "Trimethoprim/Sulfamethoxazole — oral tablets",
     "indication": "UTI; PCP prophylaxis; certain skin infections.",
     "dosage": "Adults: 1 DS tablet (160/800 mg) twice daily.",
     "contraindications": "Sulfa hypersensitivity; severe renal/hepatic impairment; pregnancy near term.",
     "warnings": "Stevens-Johnson syndrome; hyperkalemia; increased warfarin effect."},
    {"id": "pharma:031", "title": "Fluconazole — oral tablets",
     "indication": "Candidiasis; cryptococcal meningitis prophylaxis.",
     "dosage": "Adults: 150 mg single dose for vaginal candidiasis; up to 400 mg daily for systemic infections.",
     "contraindications": "Concomitant QT-prolonging CYP3A4 substrates (e.g., pimozide).",
     "warnings": "Hepatotoxicity; QT prolongation. Many CYP-mediated interactions."},
    {"id": "pharma:032", "title": "Acyclovir — oral tablets",
     "indication": "Herpes simplex; varicella zoster.",
     "dosage": "Adults: 400 mg three times daily for HSV; 800 mg five times daily for zoster.",
     "contraindications": "Hypersensitivity.",
     "warnings": "Adjust dose in renal impairment. Adequate hydration recommended to prevent crystalluria."},
    {"id": "pharma:033", "title": "Oseltamivir — oral capsules",
     "indication": "Influenza A/B treatment and prophylaxis.",
     "dosage": "Adults: 75 mg twice daily for 5 days (treatment).",
     "contraindications": "Hypersensitivity.",
     "warnings": "Most effective when started within 48 hours of symptom onset. Reports of neuropsychiatric events."},
    {"id": "pharma:034", "title": "Prednisone — oral tablets",
     "indication": "Inflammatory and autoimmune conditions.",
     "dosage": "Adults: variable; common short course 40–60 mg/day tapered.",
     "contraindications": "Systemic fungal infections; live vaccines during high-dose therapy.",
     "warnings": "Long-term: hyperglycemia, osteoporosis, adrenal suppression, infection risk. Do not discontinue abruptly after prolonged use."},
    {"id": "pharma:035", "title": "Hydrocortisone — topical cream",
     "indication": "Mild inflammatory skin conditions.",
     "dosage": "Apply thin layer 2–4 times daily.",
     "contraindications": "Skin infections without concomitant therapy; rosacea on face.",
     "warnings": "Skin atrophy with prolonged use; avoid occlusive dressings unless directed."},
    {"id": "pharma:036", "title": "Levothyroxine — oral tablets",
     "indication": "Hypothyroidism.",
     "dosage": "Adults: typical 1.6 mcg/kg/day; lower starting doses in elderly/cardiac patients.",
     "contraindications": "Untreated thyrotoxicosis; uncorrected adrenal insufficiency.",
     "warnings": "Narrow therapeutic index. Many absorption interactions (calcium, iron, PPIs). Take on empty stomach."},
    {"id": "pharma:037", "title": "Insulin glargine — subcutaneous injection",
     "indication": "Long-acting basal insulin for type 1 and type 2 diabetes.",
     "dosage": "Adults: individualized; typical starting 10 units/day or 0.2 units/kg.",
     "contraindications": "Hypoglycemia; hypersensitivity to insulin glargine.",
     "warnings": "Hypoglycemia is the most common adverse effect. Do not dilute or mix with other insulins."},
    {"id": "pharma:038", "title": "Glipizide — oral tablets",
     "indication": "Type 2 diabetes mellitus.",
     "dosage": "Adults: 5 mg once daily before breakfast; maximum 40 mg/day divided.",
     "contraindications": "Type 1 diabetes; diabetic ketoacidosis.",
     "warnings": "Hypoglycemia, especially in elderly or with skipped meals. Sulfonylurea-class cardiovascular concerns."},
    {"id": "pharma:039", "title": "Sertraline — oral tablets",
     "indication": "Major depression; anxiety disorders; PTSD; OCD.",
     "dosage": "Adults: start 25–50 mg/day; titrate to 50–200 mg/day.",
     "contraindications": "Concomitant MAOIs; pimozide.",
     "warnings": "Boxed warning: suicidality in young adults. Serotonin syndrome with serotonergic agents. QT prolongation at high doses."},
    {"id": "pharma:040", "title": "Fluoxetine — oral capsules",
     "indication": "Depression; OCD; bulimia.",
     "dosage": "Adults: start 20 mg/day; maximum 80 mg/day.",
     "contraindications": "Concomitant or recent (within 14 days) MAOI use; pimozide; thioridazine.",
     "warnings": "Long half-life; washout required before MAOI. Boxed warning: suicidality in young adults. Serotonin syndrome risk."},
    {"id": "pharma:041", "title": "Escitalopram — oral tablets",
     "indication": "Major depression; generalized anxiety disorder.",
     "dosage": "Adults: start 10 mg/day; maximum 20 mg/day.",
     "contraindications": "Concomitant MAOIs; pimozide.",
     "warnings": "QT prolongation — dose limited in elderly. Boxed warning: suicidality in young adults."},
    {"id": "pharma:042", "title": "Bupropion — extended-release",
     "indication": "Major depression; smoking cessation.",
     "dosage": "Adults: 150 mg once daily; titrate to 300 mg/day.",
     "contraindications": "Seizure disorders; current or prior anorexia/bulimia; abrupt alcohol or sedative discontinuation; MAOIs.",
     "warnings": "Dose-dependent seizure risk. Boxed warning: suicidality. Hypertension reported."},
    {"id": "pharma:043", "title": "Alprazolam — oral tablets",
     "indication": "Anxiety; panic disorder (short-term).",
     "dosage": "Adults: 0.25–0.5 mg three times daily; titrate carefully.",
     "contraindications": "Concomitant strong CYP3A4 inhibitors (e.g., ketoconazole); narrow-angle glaucoma.",
     "warnings": "Boxed warning: dependence, withdrawal, fatal respiratory depression with opioids. Avoid abrupt discontinuation."},
    {"id": "pharma:044", "title": "Lorazepam — oral tablets",
     "indication": "Anxiety; status epilepticus (IV).",
     "dosage": "Adults: 1–2 mg two or three times daily.",
     "contraindications": "Severe respiratory insufficiency; narrow-angle glaucoma.",
     "warnings": "Boxed warning: respiratory depression with opioids; dependence/withdrawal. Caution in elderly."},
    {"id": "pharma:045", "title": "Gabapentin — oral capsules",
     "indication": "Postherpetic neuralgia; partial seizures (adjunct).",
     "dosage": "Adults: 300 mg day 1, titrate up to 1800–3600 mg/day divided.",
     "contraindications": "Hypersensitivity.",
     "warnings": "Respiratory depression with opioids or in COPD. Somnolence. Adjust in renal impairment."},
    {"id": "pharma:046", "title": "Tramadol — oral tablets",
     "indication": "Moderate to moderately severe pain.",
     "dosage": "Adults: 50–100 mg every 4–6 hours; maximum 400 mg/day.",
     "contraindications": "Children under 12; post-tonsillectomy in children under 18; severe respiratory depression; concomitant MAOIs.",
     "warnings": "Boxed warnings: addiction, respiratory depression, ultra-rapid metabolizers, serotonin syndrome. Seizure risk."},
    {"id": "pharma:047", "title": "Morphine sulfate — immediate-release",
     "indication": "Moderate to severe acute pain.",
     "dosage": "Adults (opioid-naive): 15–30 mg every 4 hours as needed.",
     "contraindications": "Significant respiratory depression; acute or severe asthma without monitoring; paralytic ileus.",
     "warnings": "Boxed warnings: addiction, abuse, misuse; life-threatening respiratory depression; neonatal opioid withdrawal; risks with benzodiazepines and alcohol."},
    {"id": "pharma:048", "title": "Codeine — oral tablets",
     "indication": "Mild to moderate pain; cough (combination products).",
     "dosage": "Adults: 15–60 mg every 4–6 hours.",
     "contraindications": "Children under 12; post-tonsillectomy in children under 18; ultra-rapid CYP2D6 metabolizers.",
     "warnings": "Boxed warnings similar to other opioids. Variable metabolism leads to unpredictable effect."},
    {"id": "pharma:049", "title": "Sumatriptan — oral tablets",
     "indication": "Acute migraine.",
     "dosage": "Adults: 25–100 mg at onset; may repeat after 2 hours; maximum 200 mg/day.",
     "contraindications": "Ischemic heart disease; uncontrolled hypertension; recent ergot or other triptan use within 24 hours.",
     "warnings": "Coronary vasospasm; medication overuse headache; serotonin syndrome with SSRIs/SNRIs."},
    {"id": "pharma:050", "title": "Ondansetron — oral tablets",
     "indication": "Prevention of nausea and vomiting (chemotherapy, post-op).",
     "dosage": "Adults: 8 mg twice daily.",
     "contraindications": "Concomitant apomorphine; congenital long QT syndrome.",
     "warnings": "QT prolongation, especially with electrolyte imbalance. Serotonin syndrome possible."},
]

for d in docs:
    d["notice"] = MOCK_NOTICE  # stamp every entry with the safety disclaimer before writing to disk

print(f"Authored {len(docs)} mock drug entries.")
print(f"Sample: {docs[0]['title']}")  # spot-check the first entry to confirm the list is populated

## 2. Reshape into the integration's `Post` schema

The M8A lab's `index_helpers.py` expects records with fields: `id`, `subset`, `title`, `question_text`, `answer_text`, `text`, `license_version`, `source_url`. We map the four pharma fields into that shape so the same ingestion code path works.

In [ ]:
with open(CORPUS_PATH, "w") as f:  # open the output JSONL file for writing; overwrites any previous version
    for d in docs:                 # iterate over each of the 50 mock drug dictionaries
        combined = (
            f"INDICATION: {d['indication']}\n"              # label + indication text
            f"DOSAGE: {d['dosage']}\n"                      # label + dosage instructions
            f"CONTRAINDICATIONS: {d['contraindications']}\n"  # label + contraindication text
            f"WARNINGS: {d['warnings']}"                    # label + warning text (no trailing newline)
        )  # combined is the full structured text used for both embedding and retrieval display

        post = {
            "id":              d["id"],              # original corpus ID (e.g. "pharma:001")
            "subset":          "pharma",             # corpus subset tag; distinguishes this from Stack Exchange data in the M8A lab
            "title":           d["title"],           # human-readable drug name and form
            "question_text":   d["indication"],      # mapped to question_text so the M8A lab schema is satisfied
            "answer_text":     combined,             # full structured text returned as the retrievable answer
            "text":            f"{d['title']}\n\n{combined}",  # title + body; this concatenated field is what gets embedded
            "license_version": "n/a — mock corpus",             # placeholder; real corpora carry a license identifier
            "source_url":      "n/a (instructor-curated mock)", # placeholder; no real URL for instructor-authored mock data
            "notice":          d["notice"],          # MOCK safety disclaimer stamped in the previous cell
        }
        f.write(json.dumps(post) + "\n")  # serialise the post as compact JSON and append a newline → valid JSONL format

print(f"Wrote {CORPUS_PATH.resolve()} ({CORPUS_PATH.stat().st_size} bytes).")  # confirm file location and size

## 3. Connect to Weaviate (port 8091)

Run the container first:

```bash
docker run -d --name weaviate-si-demo-int \
  -p 8091:8080 \
  -p 50051:50051 \
  -e DEFAULT_VECTORIZER_MODULE=none \
  -e ENABLE_MODULES= \
  -e PERSISTENCE_DATA_PATH=/var/lib/weaviate \
  semitechnologies/weaviate:1.28.4
```

The second `-p` exposes the gRPC port that the v4 Python client uses on `.connect()`. Without it you get a `WeaviateStartUpError` even though `/v1/.well-known/ready` returns 200.

Readiness check: `curl -s localhost:8091/v1/.well-known/ready` should return 200.

In [ ]:
# connect to the local Weaviate instance; grpc_port is required by the v4 client for data operations
client = weaviate.connect_to_local(host=WEAVIATE_HOST, port=WEAVIATE_PORT, grpc_port=50051)
assert client.is_ready(), "Weaviate is not ready on port 8091"  # fail fast if the container isn't up yet
print("Weaviate connected and ready.")

## 4. Create the `Post` collection (idempotent)

We use `vectorizer_config=Configure.Vectorizer.none()` because we'll supply our own vectors from the local embedder (no external vectorizer module needed in Weaviate).

In [ ]:
if client.collections.exists(CLASS_NAME):      # check if the collection already exists from a previous run
    client.collections.delete(CLASS_NAME)       # drop it so we start with a clean schema and an empty vector index
    print(f"Dropped existing {CLASS_NAME} collection.")

client.collections.create(
    name=CLASS_NAME,                                     # name of the new Weaviate collection
    vectorizer_config=Configure.Vectorizer.none(),       # no built-in vectorizer; we will supply pre-computed vectors
    properties=[
        Property(name="post_id",         data_type=DataType.TEXT),  # original corpus ID (e.g. "pharma:001")
        Property(name="subset",          data_type=DataType.TEXT),  # corpus subset label ("pharma")
        Property(name="title",           data_type=DataType.TEXT),  # drug name and dosage form
        Property(name="question_text",   data_type=DataType.TEXT),  # indication text (mapped from the drug's indication field)
        Property(name="answer_text",     data_type=DataType.TEXT),  # full structured text (indication + dosage + contraindications + warnings)
        Property(name="text",            data_type=DataType.TEXT),  # title + structured text; the field that was embedded
        Property(name="license_version", data_type=DataType.TEXT),  # license metadata placeholder
        Property(name="source_url",      data_type=DataType.TEXT),  # source URL placeholder
        Property(name="notice",          data_type=DataType.TEXT),  # MOCK safety disclaimer
    ],
)
print(f"Created {CLASS_NAME} collection.")

## 5. Ingest the corpus

Embed `text` with `all-MiniLM-L6-v2` and batch-insert into Weaviate. Should take ~10 seconds for 50 docs on CPU.

In [ ]:
posts_collection = client.collections.get(CLASS_NAME)  # get a handle to the Post collection for batch operations

with open(CORPUS_PATH) as f:                            # open the JSONL file written by the reshape cell
    records = [json.loads(line) for line in f]          # parse each line into a Python dict (one record per drug)

texts   = [r["text"] for r in records]                 # extract the text field that will be embedded for each record
vectors = embedder.encode(texts, show_progress_bar=True).tolist()  # compute 384-dim embeddings for all 50 texts at once; .tolist() converts numpy → Python lists

with posts_collection.batch.dynamic() as batch:         # open a dynamic batch context; Weaviate auto-sizes batches for optimal throughput
    for record, vec in zip(records, vectors):           # pair each corpus record with its pre-computed embedding vector
        batch.add_object(
            properties={
                "post_id":         record["id"],               # original corpus ID
                "subset":          record["subset"],           # corpus subset label
                "title":           record["title"],            # drug name and form
                "question_text":   record["question_text"],    # indication text
                "answer_text":     record["answer_text"],      # full structured text
                "text":            record["text"],             # title + structured text (the field that was embedded)
                "license_version": record["license_version"],  # license placeholder
                "source_url":      record["source_url"],       # URL placeholder
                "notice":          record["notice"],           # MOCK disclaimer
            },
            vector=vec,  # supply the pre-computed embedding so Weaviate stores it in its HNSW vector index
        )

print(f"Ingested {len(records)} documents.")  # confirm all records were submitted to the batch

## 6. Verify ingestion — aggregate count should equal 50

Pre-demo checklist item. If this prints anything other than 50, fix before screen-share.

In [ ]:
agg = posts_collection.aggregate.over_all(total_count=True)  # ask Weaviate for the total number of stored objects
print(f"Documents in {CLASS_NAME}: {agg.total_count}")        # display the count for visual verification
assert agg.total_count == 50, f"Expected 50, got {agg.total_count}"  # hard-stop if ingestion was incomplete
print("Pre-demo check: PASS")  # all 50 documents confirmed present; safe to proceed with queries

## 7. The RAG pipeline

Three pieces:
1. **Retrieve** — embed the query, fetch top-`k` nearest documents from Weaviate.
2. **Generate** — build a prompt with the canonical mock-corpus disclaimer + retrieved context, run `flan-t5-base`.
3. **Score** — compute a simple **groundedness** metric: fraction of answer tokens that appear in the retrieved context.

Groundedness here is a content-token overlap proxy (stopwords removed). The integration lab uses a more rigorous version, but this is enough to show the high-vs-low pattern on screen.

In [ ]:
# common function words excluded from the groundedness calculation so the score reflects content-word overlap
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "at", "for", "with", "by", "from", "as", "that", "this", "these",
    "those", "it", "its", "if", "then", "than", "so", "do", "does", "did", "not", "no",
    "i", "you", "he", "she", "we", "they", "have", "has", "had", "can", "could", "should",
    "would", "may", "might", "will", "shall",
}

# instructs the model to stay within the retrieved context and say "I don't know" rather than hallucinate
PROMPT_TEMPLATE = (
    "Note: This is a mock pharmaceutical product information corpus for educational purposes only.\n"
    "Answer the question using only the context below. If the context does not contain the answer, "
    "say 'I don't know.'\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer:"
)

def tokenize(text: str) -> set[str]:
    # lowercase, replace non-alphanumeric characters with spaces, split on whitespace, drop stopwords
    return {t for t in "".join(c.lower() if c.isalnum() else " " for c in text).split() if t not in STOPWORDS}

def groundedness_score(answer: str, contexts: list[str]) -> float:
    answer_tokens  = tokenize(answer)                   # content-word tokens from the generated answer
    if not answer_tokens:
        return 0.0                                      # guard against division by zero for empty answers
    context_tokens = tokenize(" ".join(contexts))       # all content tokens pooled across every retrieved document
    return len(answer_tokens & context_tokens) / len(answer_tokens)  # fraction of answer tokens found in the retrieved context

def retrieve(question: str, k: int = 5) -> list[dict]:
    q_vec    = embedder.encode(question).tolist()        # embed the query using the same model used at index time
    response = posts_collection.query.near_vector(
        near_vector=q_vec,                              # find the k vectors nearest to the query vector
        limit=k,                                        # return at most k results
        return_metadata=MetadataQuery(distance=True),   # include cosine distance so we can display it
    )
    return [
        # package each hit as a lightweight dict with only the fields the pipeline needs
        {"title": obj.properties["title"], "text": obj.properties["text"], "distance": obj.metadata.distance}
        for obj in response.objects
    ]

def generate(prompt: str, max_new_tokens: int = 128) -> str:
    inputs     = gen_tokenizer(prompt, return_tensors="pt", truncation=True)   # tokenise; truncate if the prompt exceeds the model's max length
    output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy decode for deterministic output
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)        # decode token IDs back to a human-readable string

def rag_pipeline(question: str, k: int = 5) -> dict:
    hits    = retrieve(question, k=k)                               # step 1 — retrieve: vector search for the top-k relevant documents
    context = "\n\n---\n\n".join(h["text"] for h in hits)          # concatenate retrieved docs separated by a visible divider
    prompt  = PROMPT_TEMPLATE.format(context=context, question=question)  # inject the context and question into the prompt template
    raw     = generate(prompt, max_new_tokens=128)                  # step 2 — generate: produce an answer conditioned on the prompt
    contexts = [h["text"] for h in hits]                            # collect raw context strings for groundedness scoring
    return {
        "question":           question,
        "answer":             raw.strip(),                          # remove leading/trailing whitespace from the model output
        "groundedness_score": round(groundedness_score(raw, contexts), 3),  # step 3 — score: fraction of answer tokens found in context
        "retrieved": [
            # summarise each retrieved document for display; round distance to 3 decimal places
            {"title": h["title"], "distance": round(h["distance"], 3)} for h in hits
        ],
    }

def show(result: dict) -> None:
    print(f"Q: {result['question']}")                    # print the original question
    print(f"A: {result['answer']}")                      # print the generated answer
    print(f"groundedness_score: {result['groundedness_score']}")  # print the grounding score (higher = more faithful to context)
    print("Retrieved (top-k):")
    for r in result["retrieved"]:
        print(f"  - {r['title']}  (distance={r['distance']})")   # list each retrieved document and its similarity distance
    print()  # blank line to visually separate successive query outputs

print("RAG pipeline defined.")  # signals that all pipeline functions are ready before demo queries are run

## 8. Single-fact extractive queries — RAG works cleanly

These have a clean, one-sentence answer that lives in exactly one document. Expect high groundedness. **Talking point:** *"The answer tokens come from the retrieved context. This is RAG working."*

In [ ]:
# Single-fact query: the exact answer ("maximum 4 g/day") lives in one field of the acetaminophen document
# Expected: high groundedness_score because the answer tokens come directly from the retrieved context
show(rag_pipeline("What is the maximum daily dose of acetaminophen for adults?"))

In [ ]:
# Single-fact query: the starting dose lives in a single dosage field of the metformin document
# Expected: high groundedness_score; the retriever should rank metformin as the top hit
show(rag_pipeline("What is the typical metformin starting dose for type 2 diabetes?"))

## 9. Synthesis query — multi-field grounded answer

The answer requires combining the `contraindications` and `warnings` fields of the lisinopril doc. **Talking point:** *"The retriever fetched 5 documents; the generator synthesized across the fields. Groundedness stays high because the synthesis is inside the retrieved set."*

In [ ]:
# Synthesis query: the answer spans both the contraindications and warnings fields of the lisinopril document
# Expected: high groundedness_score because synthesis draws from within the retrieved set, not outside it
show(rag_pipeline("What should a patient avoid while taking lisinopril?"))

## 10. Borderline / abstention queries — the most important learning moment

Neither of these answers is in the corpus:
- Pediatric acetaminophen dosing — the doc only covers adults.
- Ibuprofen–warfarin interaction — interaction info isn't in any single doc.

Expect `flan-t5-base` to say *"I don't know"* or hedge. If it instead invents an answer, **point at the low groundedness_score** — that's the asymmetric story the demo is built around.

**Talking point:** *"In a clinical setting, this is the difference between a useful tool and a liability. The M8A integration measures both halves separately: `answer_keyword_recall_main` for attempts, `borderline_abstain_rate` for correct refusals."*

In [ ]:
# Borderline / abstention query: the corpus covers adult acetaminophen only; pediatric dosing is absent
# Expected: model says "I don't know" and groundedness_score is low — a correct, grounded abstention
show(rag_pipeline("What is the recommended dose of acetaminophen for a 6-month-old?"))

In [ ]:
# Borderline / abstention query: no single document covers the ibuprofen–warfarin interaction explicitly
# Expected: model says "I don't know"; a low groundedness_score on any invented answer exposes hallucination
show(rag_pipeline("Can ibuprofen be combined with warfarin safely?"))

## 11. Pivot to the integration

Closing beat:

> Same RAG machinery, different corpus. In the integration lab, you'll wire this against the Stack Exchange subset. Your `answer_keyword_recall_main` will land around **0.00–0.10** on the canonical config. **That's not a bug** — that's the canonical-baseline framing. Read it before you start.

Close the Weaviate client when finished.

In [ ]:
client.close()  # release the Weaviate connection and free the associated network resources
print("Weaviate client closed. Demo complete.")

## Stop the Weaviate Docker container

Cleanly stops `weaviate-si-demo-int` so the port and resources are released after the demo. The container is preserved (not removed), so the next run reuses the existing data volume — re-running the start cell at the top will `docker start` it rather than building from scratch.

If you want a fresh container next time (e.g., after editing the corpus authoring cell), run `docker rm weaviate-si-demo-int` from a terminal after this cell finishes.

In [ ]:
import subprocess  # re-imported here so this cell can run standalone without executing the top cell first

CONTAINER_NAME = "weaviate-si-demo-int"  # must match the name used in the start-container cell

# check whether the container is currently running before attempting to stop it
running = subprocess.run(
    ["docker", "ps", "--filter", f"name=^{CONTAINER_NAME}$", "--format", "{{.Names}}"],
    capture_output=True, text=True, check=False,
).stdout.strip()

if running == CONTAINER_NAME:
    print(f"Stopping {CONTAINER_NAME}…")
    subprocess.run(["docker", "stop", CONTAINER_NAME], check=True)  # sends SIGTERM; waits for a graceful shutdown
    print(f"{CONTAINER_NAME} stopped.")
else:
    # container isn't running — nothing to stop; not an error condition
    print(f"{CONTAINER_NAME} is not running; nothing to stop.")